# DeNICE V2 — Full engineering run

Shared cluster model + binary-cosine router; Conv2 coordinate protection.
Auto K, xi=0.8, fresh seed 42, Task 0–5, 20 rounds/task, 100% original TRAIN.
Every client trains on ALL its current-task TRAIN rows. No CAL/FIT/validation split.
Only after Task 5 / Round 19: each cluster runs the SAME full 34-class TEST.
Headline: Mean Cluster Accuracy / Mean Cluster Macro-F1.

Enable GPU + Internet. Set DATA_DIR if more than one original dataset is mounted.
Uses a pinned GitHub checkout; no embedded sources or artifacts.
Round checkpoints are content-addressed; each task seals one verified ZIP.
Low quality does not stop training. Correctness errors still stop.
Preserve the original dataset + execution_provenance.json + ZIPs for resume.
Large auto-K means many full-test evaluations; continuation across Kaggle sessions may be necessary.


In [ ]:
# DeNICE V2 engineering main: edit paths/resume below; do not change the locked method.
import os
from pathlib import Path
import subprocess
import sys

CODE_REF = "1d4a285f81c26f42fecc5e57ca0fb4e781408fcb"
REPOSITORY = "https://github.com/khoilv2005/FL_IL_IDS.git"
DATA_DIR = "/kaggle/input/datasets/khoilv2005/100-clients"  # Mount root; nested dataset directory is detected below
OUTPUT_DIR = "/kaggle/working/results_denice_v2_conv2_xi_0p8_seed_42"
RESUME_ROOT = ""  # Mounted previous output directory containing execution_provenance.json and ZIPs
RESUME_TASK = None
RESUME_ROUND = None  # None restores latest completed round/task-boundary continuation
OUTPUT_BUDGET_GIB = 18.0

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
code = Path("/kaggle/working/denice_v2_source")
if not code.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPOSITORY, str(code)])
    subprocess.check_call(["git", "checkout", "--detach", CODE_REF], cwd=code)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=code, text=True).strip()
if actual != CODE_REF:
    raise ValueError("Existing source checkout differs from pinned revision; use a fresh Kaggle session")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(code/"requirements-denice-v2-kaggle.txt")])
sys.path.insert(0, str(code))
import json
import torch

data_root=Path(DATA_DIR) if DATA_DIR else Path("/kaggle/input")
if (data_root/"metadata.json").is_file() and (data_root/"global_test_data.npz").is_file() and list(data_root.glob("client_*_train.npz")):
    candidates=[data_root]
else:
    candidates=[]
    for metadata in data_root.rglob("metadata.json"):
        parent=metadata.parent
        if (parent/"global_test_data.npz").is_file() and list(parent.glob("client_*_train.npz")):
            candidates.append(parent)
if len(candidates)!=1:
    raise ValueError("Set DATA_DIR explicitly. Found original dataset candidates under " + str(data_root) + ": " + str(candidates))
DATA_DIR=str(candidates[0])
data=Path(DATA_DIR)
if not (data/"global_test_data.npz").is_file():
    raise FileNotFoundError("DATA_DIR must include the entire global_test_data.npz")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before full training")
print("Pinned source:", actual, flush=True)
print("Data:",DATA_DIR,"; using 100% original client TRAIN, current task only.",flush=True)
print("DeNICE V2: auto K / xi=.8 / Conv2 coordinate / binary_cosine / seed42 / 6x20.", flush=True)
print("All TRAIN rows; batch2048; 1 epoch/round; FP32; cuda:0. FULL TEST only after Task 5 / Round 19.",flush=True)
command=[sys.executable,"-u",str(code/"tools/train_denice_v2.py"),"--data-dir",DATA_DIR,
    "--output",OUTPUT_DIR,"--device","cuda:0",
    "--output-budget-gib",str(OUTPUT_BUDGET_GIB)]
if RESUME_TASK is not None:
    command += ["--resume-task",str(RESUME_TASK)]
    if RESUME_ROOT:command += ["--resume-root",RESUME_ROOT]
    if RESUME_ROUND is not None:command += ["--resume-round",str(RESUME_ROUND)]
subprocess.check_call(command,cwd=code)
print(Path(OUTPUT_DIR,"completion.json").read_text(encoding="utf-8"),flush=True)
